In [ ]:
from math import e
import torch.nn as nn
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, f1_score
import seaborn as sns
import sys
import os

from data_pipeline import get_full_pipeline

HIDDEN = {
    "base": 128,
    "small": 64,
    "tiny": 32
}


def run_mae(freeze_encoder: bool, mode: int, size="base", batch_size=2048):
  # load data
  (loaders, scaler, le) = get_full_pipeline(mode, batch_size=batch_size)

  train_loader, val_loader, test_loader, n_features, n_classes = loaders

  # define masking function
  def mask_features(x, mask_ratio=0.3):
    mask = torch.rand(x.size(0), x.size(1), device=x.device) < mask_ratio
    x_masked = x.clone()
    x_masked[mask==1] = 0
    return x_masked, mask

  # encoder network
  class Encoder(nn.Module):
    def __init__(self, n_features, hidden_dim):
      super().__init__()
      self.net = nn.Sequential(
          nn.Linear(n_features, 4*hidden_dim),
          nn.ReLU(),
          nn.Linear(4*hidden_dim, 2*hidden_dim),
          nn.ReLU(),
          nn.Linear(2*hidden_dim, hidden_dim)
      )
    def forward(self, x_masked):
      z = self.net(x_masked)
      return z

  # decoder network
  class Decoder(nn.Module):
    def __init__(self, n_features, hidden_dim):
      super().__init__()
      self.net = nn.Sequential(
          nn.Linear(hidden_dim, 2*hidden_dim),
          nn.ReLU(),
          nn.Linear(2*hidden_dim, 4*hidden_dim),
          nn.ReLU(),
          nn.Linear(4*hidden_dim, n_features)
      )
    def forward(self, z):
      return self.net(z)

  device = "cuda" if torch.cuda.is_available() else "cpu"
  hidden_dim = HIDDEN[size]

  # MAE model
  class MAE(nn.Module):
    def __init__(self, n_features, hidden_dim):
      super().__init__()
      self.encoder = Encoder(n_features, hidden_dim)
      self.decoder = Decoder(n_features, hidden_dim)

    def forward(self, x):
      x_masked, mask = mask_features(x)
      z = self.encoder(x_masked)
      x_recon = self.decoder(z)
      return x_recon, mask

  # training loop
  model = MAE(n_features, hidden_dim).to(device)
  optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

  def mae_epoch(loader, train=True):
    if train:
      model.train()
    else:
      model.eval()

    total_err = 0
    total_mask = 0

    with torch.set_grad_enabled(train):
      for X, _ in loader:
        X = X.to(device)
        x_recon, mask = model(X)

        err = (x_recon - X) ** 2
        masked_err_sum = err[mask].sum()
        masked_count = mask.sum()

        loss = masked_err_sum / (masked_count + 1e-8)

        if train:
          optimizer.zero_grad()
          loss.backward()
          optimizer.step()

        total_err += masked_err_sum.item()
        total_mask += masked_count.item()

    return total_err / (total_mask + 1e-8)

  for epoch in range(10):
    train_recon = mae_epoch(train_loader)
    val_recon = mae_epoch(val_loader, train=False)

  # after training discard decoder
  encoder = model.encoder
  encoder.eval()

  # attach classification head
  class MAE_Classifier(nn.Module):
    def __init__(self, trained_encoder, hidden_dim, n_classes):
      super().__init__()
      self.encoder = trained_encoder
      self.fc = nn.Linear(hidden_dim, n_classes)

    def forward(self, x):
      z = self.encoder(x)
      logits = self.fc(z)
      return logits

  # train classifier normally with CrossEntropy
  criterion = nn.CrossEntropyLoss()
  classifier = MAE_Classifier(encoder, hidden_dim, n_classes).to(device)
  if freeze_encoder:
      for p in classifier.encoder.parameters():
          p.requires_grad = False
  optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-3)


  def clf_epoch(loader, train=True):
    if train:
      classifier.train()
      if freeze_encoder:
        classifier.encoder.eval()
    else:
      classifier.eval()

    total_loss = 0
    n = 0

    all_preds = []
    all_true = []

    with torch.set_grad_enabled(train):
      for X, y in loader:
        X = X.to(device)
        y = y.to(device)
        logits = classifier(X)
        loss = criterion(logits, y)

        if train:
          optimizer.zero_grad()
          loss.backward()
          optimizer.step()

        total_loss += loss.item() * y.size(0)
        n += y.size(0)

        preds = logits.argmax(dim=1).detach().cpu().numpy()
        all_preds.append(preds)
        all_true.append(y.detach().cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_true = np.concatenate(all_true)

    avg_loss = total_loss / n
    macro_f1 = f1_score(all_true, all_preds, average='macro')
    acc = accuracy_score(all_true, all_preds)
    return{
        "loss": avg_loss,
        "macro_f1": macro_f1,
        "acc": acc,
        "preds": all_preds,
        "true": all_true
    }

  train_losses = []
  val_losses = []
  train_f1s = []
  val_f1s = []
  train_accs = []
  val_accs = []

  for epoch in range(10):
    train_epoch = clf_epoch(train_loader)
    val_epoch = clf_epoch(val_loader, train=False)
    train_losses.append(train_epoch["loss"])
    val_losses.append(val_epoch["loss"])
    train_f1s.append(train_epoch["macro_f1"])
    val_f1s.append(val_epoch["macro_f1"])
    train_accs.append(train_epoch["acc"])
    val_accs.append(val_epoch["acc"])
    train_loss = train_losses[-1]
    val_loss = val_losses[-1]
    train_f1 = train_f1s[-1]
    val_f1 = val_f1s[-1]
    train_acc = train_accs[-1]
    val_acc = val_accs[-1]

  test_epoch = clf_epoch(test_loader, train=False)
  test_loss = test_epoch["loss"]
  test_f1 = test_epoch["macro_f1"]
  test_trues = test_epoch["true"]
  test_preds = test_epoch["preds"]

  accuracy = accuracy_score(test_trues, test_preds)
  precision, recall, f1, _ = precision_recall_fscore_support(test_trues, test_preds, average='macro', zero_division=0)
  cm = confusion_matrix(test_trues, test_preds)

  return {
      'accuracy': accuracy,
      'precision': precision,
      'recall': recall,
      'f1': f1,
      'train_loss': train_losses,
      'val_loss': val_losses,
      'confusion_matrix': cm,
      'history': val_losses
  }



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/content/drive/MyDrive/School Archives/Wentworth Classes/Year 2/Semester 2/Thesis 2/Code Files/data_pipeline.py:202: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(pd.to_numeric, errors='ignore')
/content/drive/MyDrive/School Archives/Wentworth Classes/Year 2/Semester 2/Thesis 2/Code Files/data_pipeline.py:202: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(pd.to_numeric, errors='ignore')
/content/drive/MyDrive/School Archives/Wentworth Classes/Year 2/Semester 2/Thesis 2/Code Files/data_pipeline.py:202: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(pd.to_numeric, errors='ignore')
/content/drive/MyDrive

MAE Epoch 1 - Train: 5805633872255.557617, Validation: 2091653623122.070801
MAE Epoch 2 - Train: 1955803633705.089600, Validation: 1861515690128.407715
MAE Epoch 3 - Train: 1757144421213.261963, Validation: 1661219133497.774414
MAE Epoch 4 - Train: 1589080246821.112305, Validation: 1563515007798.726074
MAE Epoch 5 - Train: 21570340327785.718750, Validation: 3237587162403.097168
MAE Epoch 6 - Train: 1600472623883.710205, Validation: 1441218063502.668213
MAE Epoch 7 - Train: 1431362942598.349854, Validation: 1393180403480.509277
MAE Epoch 8 - Train: 1393526246948.375000, Validation: 1383032652354.043945
MAE Epoch 9 - Train: 1365510871891.354248, Validation: 1315963597300.692871
MAE Epoch 10 - Train: 1408910276567.549561, Validation: 1315337994251.403564
Classifier Epoch 1 - Train Loss: 13913419.953471, Train F1: 0.060173, Train Acc: 0.541695, Validation Loss: 4828995.947604, Validation F1: 0.055189, Validation Acc: 0.704331
Classifier Epoch 2 - Train Loss: 11589004.266162, Train F1: 0.06